# GB Next-Day Peak Demand Forecaster
An energy-ML model. Predicts tomorrow's peak electricity demand for Great Britain from weather + calendar.

Time-ordered train/test split (no shuffling, no lookahead), a naive baseline the model must beat, and physics-informed temperature features (heating/cooling degree days).

Data: GB demand CSV you download from Elexon Insights / NESO.
Weather: Open-Meteo

In [ ]:
# 1. Setup
!pip -q install lightgbm holidays
import pandas as pd, numpy as np, requests
import matplotlib.pyplot as plt
import lightgbm as lgb
import holidays
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 2. Load your demand CSV(s)
Download 3 years of GB demand from Elexon Insights / NESO Historic Demand Data. Run the cell, pick your file(s). It prints the columns so you can map them in the next cell.

In [ ]:
from google.colab import files
uploaded = files.upload()                       # select your demand CSV(s)
raw = pd.concat([pd.read_csv(fn) for fn in uploaded.keys()], ignore_index=True)
print(raw.columns.tolist())
raw.head()

## 3. Map the columns and build daily PEAK demand
Set the three column names to match what printed above.
- NESO Historic Demand Data: `SETTLEMENT_DATE`, `SETTLEMENT_PERIOD`, `ND`
- Elexon Insights may name them differently (e.g. `settlementDate`, `demand`) - just match your file.

In [ ]:
# >>> ADJUST to your file's column names (see print above) <<<
DATE_COL   = "SETTLEMENT_DATE"
PERIOD_COL = "SETTLEMENT_PERIOD"
DEMAND_COL = "ND"                 # National Demand in MW (or "TSD")

d = raw[[DATE_COL, PERIOD_COL, DEMAND_COL]].copy()
d[DATE_COL]   = pd.to_datetime(d[DATE_COL], dayfirst=True, errors="coerce")
d[DEMAND_COL] = pd.to_numeric(d[DEMAND_COL], errors="coerce")
d = d.dropna()

# daily peak = highest half-hour that day; MW -> GW
daily = (d.groupby(d[DATE_COL].dt.date)[DEMAND_COL].max() / 1000.0)
daily.index = pd.to_datetime(daily.index)
daily = daily.sort_index().rename("peak_gw").asfreq("D")
print(daily.describe().round(2))
daily.tail()

## 4. Weather (Open-Meteo, free, no key)
London as a GB demand proxy. Upgrade later by population-weighting several cities.

In [ ]:
lat, lon = 51.51, -0.13
start = daily.index.min().date().isoformat()
end   = daily.index.max().date().isoformat()
url = ("https://archive-api.open-meteo.com/v1/archive"
       f"?latitude={lat}&longitude={lon}&start_date={start}&end_date={end}"
       "&daily=temperature_2m_mean,temperature_2m_max,temperature_2m_min"
       "&timezone=Europe%2FLondon")
w = requests.get(url, timeout=60).json()["daily"]
wx = pd.DataFrame(w); wx["time"] = pd.to_datetime(wx["time"])
wx = wx.set_index("time"); wx.columns = ["t_mean","t_max","t_min"]
wx.tail()

## 5. Features
Calendar + **physics-informed** heating/cooling degree days (the non-linear temperature response of demand) + lags (yesterday and same-day-last-week, both known before the target day, so no lookahead).

In [ ]:
df = pd.DataFrame({"peak_gw": daily}).join(wx)

# physics-informed temperature response
df["HDD"] = (15.5 - df["t_mean"]).clip(lower=0)   # heating degree days
df["CDD"] = (df["t_mean"] - 22.0).clip(lower=0)   # cooling degree days

# calendar
df["dow"]=df.index.dayofweek; df["is_we"]=(df.index.dayofweek>=5).astype(int)
df["month"]=df.index.month;   df["doy"]=df.index.dayofyear
uk = holidays.country_holidays("GB", subdiv="ENG")
df["is_hol"] = df.index.to_series().apply(lambda x: int(x in uk))

# lags (past actuals only -> legitimate)
df["lag1"]=df["peak_gw"].shift(1); df["lag7"]=df["peak_gw"].shift(7)

df = df.dropna()
FEATURES = ["t_mean","t_max","t_min","HDD","CDD","dow","is_we","month","doy","is_hol","lag1","lag7"]
df.tail()

## 6. Time-ordered split
The most important honesty step: **no shuffling.** Train on the older period, test on the most recent 180 days the model has never seen.

In [ ]:
TEST_DAYS = 180
split = df.index.max() - pd.Timedelta(days=TEST_DAYS)
train, test = df[df.index <= split], df[df.index > split]
X_tr, y_tr = train[FEATURES], train["peak_gw"]
X_te, y_te = test[FEATURES],  test["peak_gw"]
print(f"train {train.index.min().date()} -> {train.index.max().date()}  ({len(train)} days)")
print(f"test  {test.index.min().date()} -> {test.index.max().date()}  ({len(test)} days)")

## 7. Train

In [ ]:
model = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31,
                          subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)

## 8. Honest evaluation
Metrics in GW, **plus a naive baseline** (predict = yesterday's peak). If the model can't beat that, it adds nothing - that comparison is what separates real work from a tutorial toy.

In [ ]:
mae  = mean_absolute_error(y_te, pred)
rmse = mean_squared_error(y_te, pred) ** 0.5
mape = (np.abs((y_te - pred)/y_te)).mean()*100
base_mae = mean_absolute_error(y_te, X_te["lag1"])     # baseline: yesterday's peak
print(f"Model    MAE {mae:.2f} GW | RMSE {rmse:.2f} GW | MAPE {mape:.2f}%")
print(f"Baseline MAE {base_mae:.2f} GW   <- model must beat this")

fig, ax = plt.subplots(1,2, figsize=(14,4))
ax[0].plot(y_te.index, y_te, label="actual")
ax[0].plot(y_te.index, pred, label="predicted", alpha=.8)
ax[0].set_title("Test period: actual vs predicted peak (GW)"); ax[0].legend()
lims=[y_te.min(), y_te.max()]
ax[1].scatter(y_te, pred, s=10, alpha=.5); ax[1].plot(lims, lims, "k--")
ax[1].set_xlabel("actual GW"); ax[1].set_ylabel("predicted GW"); ax[1].set_title("Actual vs predicted")
plt.tight_layout(); plt.show()

pd.Series(model.feature_importances_, index=FEATURES).sort_values().plot.barh(
    figsize=(6,4), title="Feature importance"); plt.tight_layout(); plt.show()

## 9. Predict a future day (optional)
In live use you feed the **forecast** weather for tomorrow, not actuals. Fill in tomorrow's forecast temps.

In [ ]:
tom = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
r = {"t_mean":12.0, "t_max":16.0, "t_min":8.0}          # <- tomorrow's FORECAST
r["HDD"]=max(0,15.5-r["t_mean"]); r["CDD"]=max(0,r["t_mean"]-22)
r["dow"]=tom.dayofweek; r["is_we"]=int(tom.dayofweek>=5)
r["month"]=tom.month; r["doy"]=tom.dayofyear; r["is_hol"]=int(tom in uk)
r["lag1"]=daily.dropna().iloc[-1]; r["lag7"]=daily.dropna().iloc[-7]
Xnew = pd.DataFrame([r])[FEATURES]
print(f"Predicted peak demand for {tom.date()}: {model.predict(Xnew)[0]:.2f} GW")